In [1]:
# build_master_2025_US.py
import os, re, pandas as pd
from datetime import datetime, timezone
import numpy as np

In [2]:
df1= pd.read_csv(r"/Users/pardeepwalia/Desktop/untitled folder/2025 dataset us/bi_analyst.csv")
df2=pd.read_csv(r"/Users/pardeepwalia/Desktop/untitled folder/2025 dataset us/data analyst.csv")
df3=pd.read_csv(r"/Users/pardeepwalia/Desktop/untitled folder/2025 dataset us/data_architect.csv")
df4=pd.read_csv(r"/Users/pardeepwalia/Desktop/untitled folder/2025 dataset us/data_engineer.csv")
df5=pd.read_csv(r"/Users/pardeepwalia/Desktop/untitled folder/2025 dataset us/data_science.csv")
df6=pd.read_csv(r"/Users/pardeepwalia/Desktop/untitled folder/2025 dataset us/mi_engineer.csv")

In [3]:
for d in [df1,df2,df3,df4,df5,df6]:
    print(f"{d.columns}")


Index(['company', 'displayTitle', 'extractedSalary/max', 'extractedSalary/min',
       'extractedSalary/type', 'formattedLocation', 'jobDescription',
       'pubDate'],
      dtype='object')
Index(['company', 'displayTitle', 'extractedSalary/max', 'extractedSalary/min',
       'extractedSalary/type', 'formattedLocation', 'jobDescription',
       'pubDate'],
      dtype='object')
Index(['company', 'displayTitle', 'extractedSalary/max', 'extractedSalary/min',
       'extractedSalary/type', 'formattedLocation', 'jobDescription',
       'pubDate'],
      dtype='object')
Index(['company', 'displayTitle', 'extractedSalary/max', 'extractedSalary/min',
       'extractedSalary/type', 'formattedLocation', 'jobDescription',
       'pubDate'],
      dtype='object')
Index(['company', 'displayTitle', 'extractedSalary/max', 'extractedSalary/min',
       'extractedSalary/type', 'formattedLocation', 'jobDescription',
       'pubDate'],
      dtype='object')
Index(['company', 'displayTitle', 'extractedS

In [4]:
# Canonical schema 
MASTER_COLS = [
    'title','company_name','job_location','via','description',
    'extensions','description_tokens','job_basket','listed_time',
    'location_bucket','seniority_level','salary_mid_annual','SQL',
    'Python','Excel','Tableau','Power BI','Cloud','Cloud_list',
    'skill_buckets','skill_buckets_str','skill_details',
    'skill_details_str','skills_flat','salary_min_annual',
    'salary_max_annual','salary_currency','comp_type','source_file',
    'listed_year'
]

#  Shared helpers
def parse_epoch_maybe(x):
    try:
        xi = int(x)
        ts = datetime.fromtimestamp(xi/1000, tz=timezone.utc) if xi > 10**12 else datetime.fromtimestamp(xi, tz=timezone.utc)
        return pd.to_datetime(ts)
    except Exception:
        return pd.NaT

def annualize_by_type(amount, typ):
    """typ examples: HOURLY, YEARLY, WEEKLY, MONTHLY (case-insensitive)."""
    if amount is None or pd.isna(amount): return None
    try: a = float(str(amount).replace(',', ''))
    except: return None
    t = (str(typ).lower() if pd.notna(typ) else '')
    if 'year' in t:   f = 1
    elif 'month' in t: f = 12
    elif 'week' in t:  f = 52
    elif 'hour' in t or 'hr' in t: f = 2080
    else: f = 1
    return round(a * f, 2)

def infer_location_bucket(text_blob):
    t = (text_blob or "").lower()
    if "remote" in t: return "Remote"
    if "hybrid" in t: return "Hybrid"
    if "on-site" in t or "onsite" in t: return "On-site"
    return None

def infer_seniority(title):
    t = (title or "").lower()
    if re.search(r'\b(intern|internship|co[-\s]?op)\b', t): return "Intern"
    if re.search(r'\b(junior|jr\.?|entry[-\s]?level)\b', t): return "Junior"
    if re.search(r'\b(mid[-\s]?level|intermediate|mid)\b', t): return "Mid"
    if re.search(r'\b(senior|sr\.?)\b', t): return "Senior"
    if re.search(r'\b(lead|principal|staff)\b', t): return "Lead/Principal"
    if re.search(r'\b(manager|head|director|vp|vice president)\b', t): return "Manager/Director+"
    return None

# ========= 3) Your skills extractor (assumes you already defined it earlier) =========
# from previous step you have: extract_skills(df) -> adds SQL/Python/.../Cloud_list/skill_* columns
def extract_skills(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds these columns to df:
      SQL, Python, Excel, Tableau, Power BI, Cloud, Cloud_list,
      skill_buckets, skill_buckets_str, skill_details, skill_details_str, skills_flat
    Uses title + description + (optional) description_tokens.
    """
    df = df.copy()

    # ---- unified text (robust to missing / NaN) ----
    title_s = df.get('title', '').fillna('').astype(str)
    desc_s  = df.get('description', '').fillna('').astype(str)

    tok_col = df.get('description_tokens')
    if tok_col is not None:
        tokens_s = tok_col.apply(lambda x: ' '.join(x) if isinstance(x, list)
                                            else ('' if pd.isna(x) else str(x)))
    else:
        tokens_s = pd.Series([''] * len(df), index=df.index)

    text_s = (title_s + ' ' + desc_s + ' ' + tokens_s).str.lower()

    # ---- regex dictionaries ----
    # Top-level buckets (Cloud derived from cloud_rx; do NOT put cloud terms here)
    rx = {
        'SQL'     : r'\b(?:sql|mysql|postgres(?:ql)?|tsql|oracle\s+sql|snowflake)\b',
        'Python'  : r'\b(?:python|pandas|numpy|scikit-?learn|sklearn)\b',
        'Excel'   : r'\b(?:excel|v\s*look\s*up|vlookup|pivot(?:\s*table)?s?)\b',
        'Tableau' : r'\b(?:tableau)\b',
        'Power BI': r'\b(?:power\s*-?\s*bi|pbi)\b',
    }

    # Cloud subskills (providers, platforms, orchestration, infra)
    cloud_rx = {
        # Providers
        'aws'       : r'\b(?:aws|amazon web services)\b|(?<!\w)(?:s3|glue|athena|redshift)\b',
        'azure'     : r'\b(?:azure)\b|(?<!\w)(?:synapse|data\s*factory|adf)\b',
        'gcp'       : r'\b(?:gcp|google cloud)\b|(?<!\w)(?:bigquery|dataflow|pub/?sub|pubsub)\b',
        # Data platforms / engines
        'snowflake' : r'\b(?:snowflake)\b',
        'redshift'  : r'\b(?:redshift)\b',
        'bigquery'  : r'\b(?:bigquery)\b',
        'databricks': r'\b(?:databricks)\b',
        'spark'     : r'\b(?:apache\s+spark|pyspark|spark\s*sql)\b',
        # Orchestration / ELT / Streaming
        'airflow'   : r'\b(?:airflow|apache\s*airflow)\b',
        'dbt'       : r'\b(?:dbt|data build tool)\b',
        'kafka'     : r'\b(?:kafka|apache\s*kafka)\b',
        # Infra
        'docker'    : r'\b(?:docker)\b',
        'kubernetes': r'\b(?:kubernetes|k8s)\b',
        'terraform' : r'\b(?:terraform|iac|infrastructure as code)\b',
    }

    # ---- vectorized matches ----
    skill_df = pd.DataFrame(
        {k: text_s.str.contains(pat, regex=True, na=False) for k, pat in rx.items()},
        index=df.index
    )

    cloud_df = pd.DataFrame(
        {k: text_s.str.contains(pat, regex=True, na=False) for k, pat in cloud_rx.items()},
        index=df.index
    )

    # Cloud bucket + nested list
    skill_df['Cloud'] = cloud_df.any(axis=1)
    skill_df['Cloud_list'] = cloud_df.apply(lambda r: [k for k, v in r.items() if v], axis=1)

    # ---- join & lists ----
    # Drop any stale columns we’re about to overwrite
    to_drop = [
        'SQL','Python','Excel','Tableau','Power BI','Cloud','Cloud_list',
        'skill_buckets','skill_buckets_str','skill_details','skill_details_str','skills_flat'
    ]
    df = df.drop(columns=[c for c in to_drop if c in df.columns], errors='ignore')
    df = df.join(skill_df)

    # Bucket presence list
    bucket_cols = ['SQL','Python','Excel','Tableau','Power BI','Cloud']
    present_mask = df[bucket_cols].fillna(False)

    long = present_mask.stack()
    long = long[long].rename('present').reset_index()
    skills_list = long.groupby('level_0')['level_1'].agg(list)

    empty_lists = pd.Series([[]] * len(df), index=df.index)
    skills_list = skills_list.reindex(df.index).combine_first(empty_lists)

    df['skill_buckets'] = skills_list
    df['skill_buckets_str'] = df['skill_buckets'].apply(lambda L: ', '.join(L))

    # Nested details (Cloud only for now)
    cloud_lists = df['Cloud_list'].apply(lambda L: L if isinstance(L, list) else [])
    df['skill_details'] = cloud_lists.apply(lambda L: {'Cloud': L} if L else {})

    def pretty_details(d: dict) -> str:
        if not d: return ''
        return ', '.join(
            f"{k}:[{'; '.join(v)}]" if isinstance(v, list) and v else f"{k}:[]"
            for k, v in d.items()
        )

    df['skill_details_str'] = df['skill_details'].apply(pretty_details)

    # Flattened list for counting (e.g., explode later)
    def flatten_skills(i):
        buckets = df.at[i, 'skill_buckets'] if isinstance(df.at[i, 'skill_buckets'], list) else []
        clouds  = df.at[i, 'Cloud_list'] if isinstance(df.at[i, 'Cloud_list'], list) else []
        return list(buckets) + [f"Cloud:{s}" for s in clouds]

    df['skills_flat'] = pd.Index(df.index).map(flatten_skills)

    return df
# ========= 4) US standardizer (maps the US schema to MASTER_COLS) =========
def standardize_us(df_raw: pd.DataFrame, role_label: str, source_file: str, via_label="indeed") -> pd.DataFrame:
    out = pd.DataFrame(index=df_raw.index)

    # Core fields
    out['title']         = df_raw.get('displayTitle')
    out['company_name']  = df_raw.get('company')
    out['description']   = df_raw.get('jobDescription')
    out['job_location']  = df_raw.get('formattedLocation')  # keep raw as provided
    out['via']           = via_label
    out['job_basket']    = role_label
    out['extensions']    = [[] for _ in range(len(out))]    # none in this schema
    out['description_tokens'] = [[] for _ in range(len(out))]

    # Time
    out['listed_time']   = df_raw.get('pubDate', pd.Series([None]*len(df_raw))).apply(parse_epoch_maybe)
    out['listed_year']   = pd.to_datetime(out['listed_time'], errors='coerce').dt.year

    # Salary (annualize using extractedSalary/type)
    s_min = df_raw.get('extractedSalary/min')
    s_max = df_raw.get('extractedSalary/max')
    s_typ = df_raw.get('extractedSalary/type')  # 'HOURLY','YEARLY',...
    out['salary_min_annual'] = [annualize_by_type(m, t) for m,t in zip(s_min, s_typ)]
    out['salary_max_annual'] = [annualize_by_type(m, t) for m,t in zip(s_max, s_typ)]
    out['salary_currency']   = 'USD'  # US dataset
    out['salary_mid_annual'] = pd.to_numeric(out[['salary_min_annual','salary_max_annual']].mean(axis=1), errors='coerce')

    # Buckets
    out['location_bucket'] = [infer_location_bucket(" ".join(map(str, x))) for x in zip(out['title'], out['description'], out['job_location'])]
    out['seniority_level'] = out['title'].apply(infer_seniority)

    # Skills
    out = extract_skills(out)

    # Extra metainfo
    out['comp_type']   = None            # not available in this schema (no jobTypes/*)
    out['source_file'] = source_file

    # Enforce schema/order
    for c in MASTER_COLS:
        if c not in out.columns:
            out[c] = None
    return out[MASTER_COLS]

# ========= 5) Batch runner (US files) =========
# Map your US filenames to role labels (edit filenames if needed)
ROLE_FOR_US = {
    "bi_analyst.csv"         : "Business Intelligence Analyst",
    "data analyst.csv"  : "Data Analyst",
    "data_architect.csv": "Data Architect",
    "data_engineer.csv" : "Data Engineer",
    "data_science.csv"   : "Data Scientist",
    "mi_engineer.csv"   : "Machine Learning Engineer",
}
# Base directory containing the US CSVs
base_dir_us = "/Users/pardeepwalia/Desktop/untitled folder/2025 dataset us"   

def load_and_standardize_us(file, role_label):
    path = os.path.join(base_dir_us, file)
    usecols = [
        'company','displayTitle',
        'extractedSalary/max','extractedSalary/min','extractedSalary/type',
        'formattedLocation','jobDescription','pubDate'
    ]
    df_raw = pd.read_csv(path, usecols=usecols)
    return standardize_us(df_raw, role_label, source_file=file)

frames = []
for fname, role in ROLE_FOR_US.items():
    print(f"processing (US) {fname} as {role}")
    frames.append(load_and_standardize_us(fname, role))

master_us = pd.concat(frames, ignore_index=True)

# De-dupe (same rule as CA)
dedupe_key = (
    master_us['title'].fillna('').str.lower() + '|' +
    master_us['company_name'].fillna('').str.lower() + '|' +
    master_us['listed_time'].astype(str)
)
master_us = master_us.loc[~dedupe_key.duplicated()].reset_index(drop=True)

print("US master shape:", master_us.shape)



processing (US) bi_analyst.csv as Business Intelligence Analyst
processing (US) data analyst.csv as Data Analyst
processing (US) data_architect.csv as Data Architect
processing (US) data_engineer.csv as Data Engineer
processing (US) data_science.csv as Data Scientist
processing (US) mi_engineer.csv as Machine Learning Engineer
US master shape: (1472, 30)


In [9]:
master_us['location_bucket'].value_counts()

location_bucket
Remote     337
Hybrid     210
On-site    185
Name: count, dtype: int64

In [15]:
# 
# Clearing the null values
cond1 = master_us['job_location'].str.contains(
    r'\b(?:alabama|AL|alaska|AK|arizona|AZ|arkansas|AR|california|CA|colorado|CO|connecticut|CT|delaware|DE|florida|FL|georgia|GA|hawaii|HI|idaho|ID|illinois|IL|indiana|IN|iowa|IA|kansas|KS|kentucky|KY|louisiana|LA|maine|ME|maryland|MD|massachusetts|MA|michigan|MI|minnesota|MN|mississippi|MS|missouri|MO|montana|MT|nebraska|NE|nevada|NV|new hampshire|NH|new jersey|NJ|new mexico|NM|new york|NY|north carolina|NC|north dakota|ND|ohio|OH|oklahoma|OK|oregon|OR|pennsylvania|PA|rhode island|RI|south carolina|SC|south dakota|SD|tennessee|TN|texas|TX|utah|UT|vermont|VT|virginia|VA|washington|WA|west virginia|WV|wisconsin|WI|wyoming|WY|district of columbia|DC|wyoming|WY|district of columbia|DC|puerto rico|PR)\b',
    case=False,
    na=False,
    regex=True
)

cond2 = master_us['location_bucket'].str.contains('Remote',case=False,regex=False)

cond3=master_us['job_location'].str.contains(r'\b(?:remote)\b',case=False,na=False,
        regex=True
    )
master_us['location_bucket']= np.select([cond1.to_numpy(dtype=bool),cond2.to_numpy(dtype=bool),cond3.to_numpy(dtype=bool)],['USA','Remote','Remote'],default="")

In [16]:
master_us['location_bucket'].value_counts()

location_bucket
USA       1393
Remote      79
Name: count, dtype: int64

In [18]:
master_us.to_csv(os.path.join(base_dir_us, "master_2025_US.csv"), index=False)

In [19]:
master_us.sample(10)

,title,company_name,job_location,via,description,extensions,description_tokens,job_basket,listed_time,location_bucket,...,skill_buckets_str,skill_details,skill_details_str,skills_flat,salary_min_annual,salary_max_annual,salary_currency,comp_type,source_file,listed_year
782,Data Analytics Engineer,Sunderstorm Inc,"North Hollywood, CA 91602",indeed,About Us\nSunderstormis a premier cannabis com...,[],[],Data Engineer,2025-07-14 05:13:20+00:00,USA,...,"SQL, Python, Excel, Tableau, Cloud","{'Cloud': ['aws', 'redshift']}",Cloud:[aws; redshift],"[SQL, Python, Excel, Tableau, Cloud, Cloud:aws...",80000.00,100000.00,USD,None,data_engineer.csv,2025
1049,Data Scientist Level 2- TS/SCI with Poly,DigiFlight,"Fort Meade, MD 20755",indeed,Join an outstanding team that offers exciting ...,[],[],Data Scientist,2024-05-30 06:20:00+00:00,USA,...,Python,{},,[Python],NaN,NaN,USD,None,data_science.csv,2024
161,BI Developer/Analyst for Meat Packaging Company,New Angus Llc,"Aberdeen, SD 57401",indeed,Position Overview\n We are seeking a skilled P...,[],[],Business Intelligence Analyst,2025-08-22 05:00:00+00:00,USA,...,"SQL, Python, Excel, Power BI, Cloud",{'Cloud': ['azure']},Cloud:[azure],"[SQL, Python, Excel, Power BI, Cloud, Cloud:az...",NaN,NaN,USD,None,bi_analyst.csv,2025
1118,Decision Analytics Associate Consultant - Mark...,ZS,"Boston, MA 02108",indeed,: \n \n ZS is a place where passion changes ...,[],[],Data Scientist,2025-07-21 03:53:20+00:00,USA,...,"Excel, Tableau",{},,"[Excel, Tableau]",NaN,NaN,USD,None,data_science.csv,2025
898,"Data Center Project Engineer - Holy Ridge, LA",Leapros Skilled Trades,"Fremont, CA 94538",indeed,Leapros Skilled Trades is actively hiring Data...,[],[],Data Engineer,2025-08-16 04:53:20+00:00,USA,...,Excel,{},,[Excel],105000.00,105000.00,USD,None,data_engineer.csv,2025
1299,Software Developer Expert - TS/SCI w/Polygraph,General Dynamics Information Technology,"Herndon, VA 20170",indeed,Type of Requisition: Regular\n \n Clearance L...,[],[],Machine Learning Engineer,2025-07-31 05:00:00+00:00,USA,...,"Python, Cloud",{'Cloud': ['aws']},Cloud:[aws],"[Python, Cloud, Cloud:aws]",159800.00,216200.00,USD,None,mi_engineer.csv,2025
1286,Systems Engineer (AI/ML),Peraton,"Springfield, VA",indeed,About Peraton Peraton is a next-generation nat...,[],[],Machine Learning Engineer,2025-08-07 05:00:00+00:00,USA,...,Tableau,{},,[Tableau],135000.00,216000.00,USD,None,mi_engineer.csv,2025
802,Sr. Data Engineer (AWS/Databricks),Onebridge,"New York, NY 10017",indeed,"Onebridge, a Marlabs Company, is a global AI a...",[],[],Data Engineer,2025-09-03 06:13:20+00:00,USA,...,"SQL, Cloud","{'Cloud': ['aws', 'redshift', 'databricks', 's...",Cloud:[aws; redshift; databricks; spark],"[SQL, Cloud, Cloud:aws, Cloud:redshift, Cloud:...",135000.81,162581.61,USD,None,data_engineer.csv,2025
904,Data Center Chief Engineer,"Amazon Data Services, Inc.","Hermiston, OR 97838",indeed,"AWS Infrastructure Services owns the design, p...",[],[],Data Engineer,2025-07-18 03:40:00+00:00,USA,...,"Excel, Cloud",{'Cloud': ['aws']},Cloud:[aws],"[Excel, Cloud, Cloud:aws]",83100.00,185000.00,USD,None,data_engineer.csv,2025
1267,Staff Engineer - GM Energy,General Motors,"Roswell, GA",indeed,Job Description \n This role is categorized as...,[],[],Machine Learning Engineer,2025-08-22 05:00:00+00:00,USA,...,"SQL, Python, Cloud","{'Cloud': ['aws', 'azure', 'gcp', 'snowflake',...",Cloud:[aws; azure; gcp; snowflake; databricks;...,"[SQL, Python, Cloud, Cloud:aws, Cloud:azure, C...",165000.00,270900.00,USD,None,mi_engineer.csv,2025
